# **Data Cleaning**

#### **Customers**

In [ ]:
-- Cleaning Table Customers

GO
CREATE OR ALTER VIEW cl_customers AS
WITH 
	normalization as (
		SELECT 
			customer_id,

			-- Normalisasi Penulisan Kapital
			TRIM(STRING_AGG(UPPER(LEFT(value, 1)) + lower(SUBSTRING(value, 2)), ' ' )) customer_name, 
			
			-- Penyeragaman Penulisan Istilah
			max( 
				CASE 
					WHEN UPPER(gender) = 'F' THEN 'Female'
					WHEN UPPER(gender) = 'M' THEN 'Male'
					ELSE gender
				END
			) gender,

			-- Normalisasi format tanggal
			max(
				FORMAT(birth_date, 'dd-MM-yyyy')
			) birth_date,
			
			-- Penyeragaman penulisan nomor telepon
			max(
		
				CASE 
					WHEN LEFT(REPLACE(phone, '-', ''), 2) = '62' THEN '0' + SUBSTRING(phone, 3)
					WHEN LEFT(REPLACE(phone, '-', ''), 3) = '+62' THEN '0' + SUBSTRING(phone, 4)
					ELSE REPLACE(phone, '-', '')
				END
		
			) phone,

			-- Pemberian keterangan pada email
			max(COALESCE(email, '-')) email,
			max(TRIM(city)) city,
			max(TRIM(province)) province,

			-- Normalisasi format tanggal
			max(
				FORMAT(join_date, 'dd-MM-yyyy')
			) join_date,
			
			--- Perbaikan kesalahan penulisan
			max(
				case 
					when lower(customer_segment) = 'membre' then 'member'
					when lower(customer_segment) = 'regular' then 'reguler'
					else lower(customer_segment)
				end
			)  customer_segment


		FROM customers
		CROSS APPLY string_split(customer_name, ' ', 1)
		GROUP BY customer_id
		)

select * from normalization;

#### **Order Details**

In [ ]:
-- Cleaning Table Order Details

GO
CREATE OR ALTER VIEW cl_order_details AS
WITH 
	cleaned_order_details AS (
		select 
			od.detail_id, 
			od.order_id, 
			od.product_id,
			od.quantity,
			od.unit_price,
			od.discount_rate,
			od.line_total,
			ROW_NUMBER() OVER(PARTITION BY o.customer_id, od.product_id ORDER BY o.order_date) AS ProductOrderNumber
		from order_details od
		LEFT JOIN orders o on o.order_id = od.order_id
	)

select * from cleaned_order_details;

#### **Orders**

In [ ]:
-- Cleaning Table Orders

GO
CREATE OR ALTER VIEW cl_orders AS
WITH
    orders_cleaning AS (
        SELECT
            o.order_id,
            o.customer_id,
            o.branch_name,
            o.cashier_name,
            o.order_date,
            o.order_time,
            CASE 
                WHEN payment_method IN ('Kartu Debit', 'Debit') THEN 'Debit Card'
                WHEN payment_method = 'Kartu Kredit' THEN 'Credit Card'
                WHEN payment_method = 'Tunai' THEN 'Cash'
                WHEN lower(payment_method) IN ('ovo', 'shopeepay', 'gopay', 'qr code', 'qris') THEN 'E-Wallet'
                ELSE (
                        SELECT STRING_AGG(UPPER(LEFT(value, 1)) + SUBSTRING(lower(value), 2), ' ')
                        FROM string_split(o.payment_method, ' ', 1)
                    )
            END AS payment_method,
            REPLACE(
                REPLACE(
                    COALESCE(o.promotion, '-'), '-', 'Tanpa Promo'), 
                    'NONE', 'Tanpa Promo'
                ) promotion,
            (
                SELECT
                    SUM(quantity * unit_price)
                    FROM cl_order_details cod
                    WHERE cod.order_id = o.order_id
            ) subtotal,
            o.tax,
            o.tax_rate
        FROM orders o
        WHERE EXISTS(
            SELECT 1 FROM cl_order_details cod2
            WHERE cod2.order_id = o.order_id
        )
    ),

    orders_engineering AS (
        select 
            oc2.*,
            ROW_NUMBER() OVER(PARTITION BY oc2.customer_id ORDER BY oc2.order_date) AS CustomerOrderNumber,
	        ROW_NUMBER() OVER(PARTITION BY oc2.customer_id, oc2.branch_name ORDER BY oc2.order_date) AS OrderInBranchNumber 
        from orders_cleaning oc2
    )

SELECT *
FROM orders_engineering;

#### **Products**

In [ ]:
-- Cleaning Table Products

GO
CREATE OR ALTER VIEW cl_products AS
select 
	product_id,
	product_name,
	category,

	-- Ada beberapa harga beli lebih besar dari harga jual (kemungkinan tertukar)
	CASE 
		WHEN cost_price > selling_price THEN selling_price
		ELSE cost_price
	END AS cost_price,
	CASE 
		WHEN cost_price > selling_price THEN cost_price
		ELSE selling_price
	END AS selling_price,

	-- Normalisasi format tanggal
	FORMAT(launch_date, 'dd-MM-yyyy') as launch_date,

	-- Normalisasi Penulisan
	UPPER(LEFT(status, 1)) + SUBSTRING(lower(status), 2) as status

from products;